# 🧑‍💻 Developer vs Tester Agents: a Critique-and-Revise Loop

Two agents improve a piece of code together. The **developer** writes Python (and can run it with a Python tool);
the **tester** reviews it, writes unit tests in its report, scores it and lists what to fix. The developer
revises, and the loop repeats for a fixed number of rounds. We then point the pair at two small games:
**FLAMES** and **Tic-Tac-Toe**.

![Works on my machine](../assets/11_works_on_my_machine.png)

## Learning Objectives
1. **The critique-and-revise pattern** - A generator and a reflector take turns on one shared conversation
2. **A Python tool for model-written code** - Running it in a separate process with a timeout and no keyboard
   input (a guard, not a sandbox)
3. **Developer and tester nodes** - A tool-using agent (`create_agent`) and a plain prompt-and-model reviewer
4. **Loop termination in code** - A reflection counter in state, a pure routing function and a `recursion_limit`,
   proven with stub nodes and zero model calls
5. **Applying the loop** - Building FLAMES and Tic-Tac-Toe, with hand-written reference implementations checked in code

## Where this fits
`10_hotel_web_search_integration.ipynb` finished the hotel build. This notebook starts the second half of the
module: smaller, self-contained multi-agent applications. `12_supervisor_financial_research.ipynb` follows with a
supervisor that delegates to a researcher and a coder.

## Prerequisites
- Multi-agent basics from `modules/05_multi_agent_foundations/` (reading only; this notebook defines everything it uses)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks used below.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import ast
import operator
import re
import time
from typing import Annotated

from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the developer and tester agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🔁 Part 1: The Critique-and-Revise Pattern

![Critique & Revise Design Pattern](../assets/11_critique_revise_title.png)

![The critique-and-revise loop](../assets/11_critique_revise_loop.png)

1. The user's request goes to the **generator** (our developer).
2. It produces an initial response (code).
3. The **reflector** (our tester) critiques it: what works, what fails, what to change.
4. The critique becomes a new message in the conversation.
5. The generator revises, seeing its previous draft *and* the critique, until a fixed number of rounds is done.

The loop must end by a rule written in code, not by a model deciding it is happy. Here that rule is a
reflection counter in the graph state.

---
## 🐍 Part 2: A Python Tool for the Developer

Sometimes it is better to have the model *write and run* code than to predict an answer. The developer's
`python_repl` tool calls `run_python()` from `python_runner.py`, a small module next to this notebook
(`12_supervisor_financial_research.ipynb` uses it too); the cell prints its source. For every snippet it:
- writes the code to a file in a fresh, empty temporary folder and runs it there in a **new Python process**
  (`sys.executable -I`, isolated mode) whose environment is **empty**, so no API keys reach it;
- **closes standard input**: a game that waits for a player gets `EOFError` instead of blocking;
- **kills the process after `timeout` seconds**: an infinite game loop cannot hang the notebook;
- runs the snippet as one script, so a function defined in it can call another one defined next to it.

> ⚠️ **Safety note:** this tool runs code *written by a model* on your machine. A child process with a timeout is
> **not a sandbox**: the code still runs with your user's permissions, so it can read and write your files and
> reach the network. Run it only in a throwaway environment, and use a container or a hosted sandbox
> ([deep agents sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes))
> for anything beyond a demo. Here the code comes from your own task text;
> `12_supervisor_financial_research.ipynb` shows the extra step needed when text from the web can reach the agent
> that runs code.

In [ ]:
# ============================================================================
# A TIMED, NON-INTERACTIVE PYTHON TOOL
# ============================================================================
import inspect

from python_runner import run_python  # in this folder; 12_supervisor_financial_research.ipynb uses it too

PYTHON_TIMEOUT_S = 10

print(inspect.getsource(run_python))


@tool
def python_repl(code: str) -> str:
    """Run Python code in a fresh subprocess and return what it printed (use print(...) to show values).
    Each run starts empty, so include every definition the code needs. Keyboard input is not available."""
    return run_python(code, timeout=PYTHON_TIMEOUT_S)


print(python_repl.invoke({"code": "def square(x):\n    return x * x\n\ndef report():\n    print([square(n) for n in range(5)])\n\nreport()"}))

# The guards, checked without a model: the same function the tool calls, with a 2-second timeout
asked = run_python("name = input('Your name? ')")
start = time.perf_counter()
looped = run_python("while True:\n    pass", timeout=2)
elapsed = time.perf_counter() - start
print(f"{'✅' if 'EOFError' in asked else '❌'} input() failed at once instead of waiting: {asked.splitlines()[-1]}")
print(f"{'✅' if looped.startswith('Timed out') and elapsed < 2 + 5 else '❌'} the endless loop was stopped "
      f"after {elapsed:.1f}s: {looped}")

**Reading the output:** the source of `run_python` comes first. The first snippet prints `[0, 1, 4, 9, 16]`
(`report` found `square` in the same script). The two ✅ lines check the guards in code: `input()` failed at once
with `EOFError`, and the endless loop was killed after about 2 seconds (the developer's tool allows 10).

---
## 👩‍💻 Part 3: The Developer Agent

The developer is a tool-using agent built with `create_agent` (the LangChain 1.x replacement for the old
`AgentExecutor`). It is built **once** and reused by the graph node. The node passes the whole conversation,
so on a revision round the developer sees its previous code *and* the tester's report. Its own tool loop is
capped with a `recursion_limit` of 12 (a handful of tool calls).

In [ ]:
# ============================================================================
# DEVELOPER AGENT AND NODE
# ============================================================================
DEVELOPER_PROMPT = """You are an intelligent coder who is well trained in writing Python code for the scenario given.
Your job is to write Python code and refine it based on the suggestions and unit test results you receive.
Generate the best Python code for the user's requirement, covering the edge cases you can think of.
If the conversation contains a critique or a unit test report, respond with a revised version of your previous attempt that takes the recommendations into account.
You may run code with the python_repl tool. When you do, test your functions with fixed example arguments: never read keyboard input or start an interactive game loop in the tool.
Generate the COMPLETE answer at every iteration: put the full, final program in a single ```python code block."""

developer_agent = create_agent(llm, tools=[python_repl], system_prompt=DEVELOPER_PROMPT)


def developer_node(state: dict) -> dict:
    """Write (or revise) the code, seeing the whole developer/tester conversation."""
    result = developer_agent.invoke({"messages": state["conversation_history"]}, {"recursion_limit": 12})
    answer = result["messages"][-1].content
    tool_runs = sum(1 for m in result["messages"] if m.type == "tool")
    display(Markdown(f"**👩‍💻 DEVELOPER** (ran the Python tool {tool_runs}×)\n\n{answer}"))
    return {"conversation_history": [AIMessage(content=answer, name="developer")]}


print(f"✅ Developer agent ready with tools: {[t.name for t in [python_repl]]}")

---
## 🧪 Part 4: The Tester Agent

The tester is a plain prompt piped into the model: no tools. It writes unit tests *in its report* and reasons
about whether they pass, so its pass percentage and score are the model's **estimates**, not executed results
(giving it the Python tool too would make them real, at roughly twice the cost). Its report is added to the
conversation as a `HumanMessage`, because from the developer's point of view it is feedback from a user.

In [ ]:
# ============================================================================
# TESTER AGENT AND NODE
# ============================================================================
TESTER_PROMPT = """You are an intelligent tester who is well trained in writing exhaustive unit test cases for the given Python code.
Write the unit tests, work out which ones pass and give a detailed summary of the unit testing results.
Recommend how to fix every failing test.
At every submission, score the code from 0-10 and state the percentage of unit tests passed.
The score reflects the percentage of tests passed and the quality of the code, including docstrings and comments.
End with a critique if any tests failed."""

tester_chain = ChatPromptTemplate.from_messages([
    ("system", TESTER_PROMPT),
    MessagesPlaceholder(variable_name="messages"),
]) | llm


def tester_node(state: dict) -> dict:
    """Review the latest code and add one reflection to the counter."""
    round_no = state["reflection_count"] + 1
    reflection = tester_chain.invoke({"messages": state["conversation_history"]})
    display(Markdown(f"**🧪 TESTER** (reflection {round_no} of {state['max_reflections']})\n\n{reflection.content}"))
    return {"conversation_history": [HumanMessage(content=reflection.content, name="tester")], "reflection_count": 1}


print("✅ Tester chain ready")

---
## 🗺️ Part 5: Shared State and the Loop

The state holds the conversation, the reflection cap and a counter. `reflection_count` uses `operator.add` as its
reducer, so each tester turn adds 1. After every developer turn a **pure** routing function (it only reads state)
decides: go to the tester, or stop because the cap is reached. The graph builder takes the two nodes as
arguments, so the next step can prove the loop logic with stubs.

In [ ]:
# ============================================================================
# STATE, ROUTING FUNCTION AND GRAPH BUILDER
# ============================================================================
class GraphState(TypedDict):
    """max_reflections: how many tester rounds to allow; reflection_count: rounds done so far;
    conversation_history: the developer/tester conversation."""
    max_reflections: int
    reflection_count: Annotated[int, operator.add]
    conversation_history: Annotated[list[AnyMessage], operator.add]


def should_continue(state: GraphState) -> str:
    """Pure routing: stop once the tester has reflected max_reflections times."""
    if state["reflection_count"] >= state["max_reflections"]:
        return END
    return "tester_node"


def build_reflection_graph(developer, tester):
    builder = StateGraph(GraphState)
    builder.add_node("developer_node", developer)
    builder.add_node("tester_node", tester)
    builder.add_edge(START, "developer_node")
    builder.add_conditional_edges("developer_node", should_continue, ["tester_node", END])
    builder.add_edge("tester_node", "developer_node")
    return builder.compile()


graph = build_reflection_graph(developer_node, tester_node)
print("✅ Graph compiled. Edges:")
for edge in graph.get_graph().edges:
    print(f"   {edge.source} -> {edge.target}{' (conditional)' if edge.conditional else ''}")

---
## ✅ Part 6: Proving the Loop Ends (No Model Calls)

With `max_reflections = r` the developer should run `r + 1` times (a first draft plus one revision per critique)
and the tester `r` times. Stub nodes that only append a message let us check that for several caps. The run
helper also passes a `recursion_limit` of `2r + 4` graph steps, a second guard in case the routing were ever broken.

In [ ]:
# ============================================================================
# STUB NODES PROVE THE CAP
# ============================================================================
def stub_developer(state):
    return {"conversation_history": [AIMessage(content="code", name="developer")]}


def stub_tester(state):
    return {"conversation_history": [HumanMessage(content="critique", name="tester")], "reflection_count": 1}


def reflection_config(max_reflections: int) -> dict:
    return {"recursion_limit": 2 * max_reflections + 4}


stub_graph = build_reflection_graph(stub_developer, stub_tester)
for r in (0, 1, 3):
    final = stub_graph.invoke(
        {"conversation_history": [HumanMessage(content="task")], "max_reflections": r, "reflection_count": 0},
        reflection_config(r),
    )
    names = [m.name for m in final["conversation_history"]]
    dev, test = names.count("developer"), names.count("tester")
    ok = dev == r + 1 and test == r
    print(f"{'✅' if ok else '❌'} max_reflections={r}: developer ran {dev}×, tester ran {test}×")

---
## ▶️ Part 7: The Run Helper

`run_reflection` starts the loop on a task and returns the final state. It then checks two things in code: how
many times each agent spoke, and whether the developer's last answer contains a Python code block that parses
(`ast.parse` only reads the code, it does not run it). One reflection round per task keeps the cost small:
raise `max_reflections` to watch more revisions.

In [ ]:
# ============================================================================
# RUN ONE CRITIQUE-AND-REVISE LOOP
# ============================================================================
def last_code_block(text: str) -> str | None:
    blocks = re.findall(r"```python\n(.*?)```", text, re.S)
    return blocks[-1] if blocks else None


def run_reflection(graph, question: str, max_reflections: int = 1) -> dict:
    inputs = {"conversation_history": [HumanMessage(content=question)],
              "max_reflections": max_reflections, "reflection_count": 0}
    final = graph.invoke(inputs, reflection_config(max_reflections))

    names = [m.name for m in final["conversation_history"][1:]]
    print(f"\n{'✅' if names.count('tester') == max_reflections else '❌'} tester reflections: {names.count('tester')} "
          f"(cap {max_reflections}); developer turns: {names.count('developer')}")
    code_text = last_code_block(final["conversation_history"][-1].content)
    try:
        ast.parse(code_text or "")
        parsed = code_text is not None
    except SyntaxError:
        parsed = False
    print(f"{'✅' if parsed else '❌'} final answer contains a Python code block that parses")
    return final


print("✅ run_reflection ready")

---
## ❤️ Part 8: Build the FLAMES Game

**What is FLAMES?** A paper game that "predicts" the relationship between two people from their names.

1. Take two names, for example **John** and **Jane**.
2. **Remove common letters**, once per occurrence (here J and N), and count what is left: o, h, a, e → **4**.
3. Count round the word **FLAMES** (Friends, Love, Affection, Marriage, Enemy, Sibling) using that number and
   cross out the letter you land on: F(1), L(2), A(3), **M(4)** → remove M.
4. Keep counting from the next letter, round and round, until one letter is left. That letter is the "result".

In [ ]:
# ============================================================================
# DEVELOPER AND TESTER BUILD FLAMES (1 REFLECTION ROUND)
# ============================================================================
flames_state = run_reflection(
    graph,
    "Write a Python program for the FLAMES game: a function that takes two names and returns the relationship, "
    "plus a short example that calls it with two fixed names.",
    max_reflections=1,
)

**Reading the output:** compare the tester's report with the developer's second answer: which of the tester's
recommendations were applied? Check the two ✅ lines at the end. Remember the tester's pass rate is its own
estimate; the reference implementation below is checked in code.

### 8.1 A reference implementation

The version below is written by hand and avoids two mistakes a model run made: it continues counting from the
**next** letter after every removal (not from the start of FLAMES), and `S` means "Sibling", not "Sister". It
runs on fixed examples, so the cell runs unattended. To play, call `flames_game("your name", "their name")` in a
new cell.

In [ ]:
# ============================================================================
# FLAMES REFERENCE IMPLEMENTATION, CHECKED ON THE WORKED EXAMPLE
# ============================================================================
from collections import Counter

FLAMES_MEANING = {"F": "Friends", "L": "Love", "A": "Affection", "M": "Marriage", "E": "Enemy", "S": "Sibling"}


def remaining_letter_count(name1: str, name2: str) -> int:
    """Letters left after crossing out common letters once per occurrence."""
    a = Counter(ch for ch in name1.lower() if ch.isalpha())
    b = Counter(ch for ch in name2.lower() if ch.isalpha())
    return sum(((a - b) + (b - a)).values())


def flames_game(name1: str, name2: str) -> tuple[str, list[str]]:
    """Return the final FLAMES letter and the order in which letters were crossed out."""
    if not isinstance(name1, str) or not isinstance(name2, str):
        raise ValueError("Both inputs must be strings.")
    count = remaining_letter_count(name1, name2)
    if count == 0:
        raise ValueError("The names cancel out completely: no letters left to count.")
    letters, removed, position = list("FLAMES"), [], 0
    while len(letters) > 1:
        position = (position + count - 1) % len(letters)  # keep counting from where the last letter was removed
        removed.append(letters.pop(position))
    return letters[0], removed


result, crossed_out = flames_game("John", "Jane")
print(f"John + Jane: {remaining_letter_count('John', 'Jane')} letters left, crossed out {crossed_out} "
      f"-> {FLAMES_MEANING[result]}")
print(f"{'✅' if remaining_letter_count('John', 'Jane') == 4 and crossed_out[0] == 'M' else '❌'} "
      "matches the worked example (4 letters, M crossed out first)")

for pair in [("Alice", "Bob"), ("Romeo", "Juliet")]:
    letter, _ = flames_game(*pair)
    print(f"{pair[0]} + {pair[1]} -> {FLAMES_MEANING[letter]}")

---
## ❌⭕ Part 9: Build Tic-Tac-Toe

Same loop, a different task. A Tic-Tac-Toe program normally loops on player input, which is exactly what the
Python tool refuses to run, so the task asks for a game that can also be driven by a list of moves.

In [ ]:
# ============================================================================
# DEVELOPER AND TESTER BUILD TIC-TAC-TOE (1 REFLECTION ROUND)
# ============================================================================
tictactoe_state = run_reflection(
    graph,
    "Write a Python program for a two-player Tic-Tac-Toe game. Include a function that plays a game from a list "
    "of moves (positions 1-9) and reports the winner or a draw, and demonstrate it on one fixed list of moves.",
    max_reflections=1,
)

**Reading the output:** look for how the developer handles an occupied square and a full board, and whether the
tester's critique mentions them. The ✅ lines report the loop count and whether the final code parses.

### 9.1 A reference implementation

The hand-written class below is a reference for what the pair should build. `play_moves()` takes a scripted
list of moves instead of reading the keyboard, positions are 1-9 as the prompt asks, and no move is accepted
after the game is over. Three scripted games
check a win, a draw and an illegal move.

In [ ]:
# ============================================================================
# TIC-TAC-TOE REFERENCE IMPLEMENTATION, THREE SCRIPTED GAMES
# ============================================================================
class TicTacToe:
    WIN_LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
                 (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columns
                 (0, 4, 8), (2, 4, 6)]              # diagonals

    def __init__(self):
        self.board = [" "] * 9
        self.current_player = "X"
        self.winner = None
        self.over = False

    def board_text(self) -> str:
        rows = [" " + " | ".join(self.board[i:i + 3]) for i in (0, 3, 6)]
        return "\n---+---+---\n".join(rows)

    def check_winner(self):
        for a, b, c in self.WIN_LINES:
            if self.board[a] == self.board[b] == self.board[c] != " ":
                return self.board[a]
        return None

    def make_move(self, position: int) -> str:
        """Play `position` (1-9) for the current player and describe the result."""
        if self.over:
            return "The game is already over."
        if not 1 <= position <= 9:
            return "Invalid position. Please choose a number between 1 and 9."
        if self.board[position - 1] != " ":
            return "Invalid move. The position is already occupied."
        self.board[position - 1] = self.current_player
        self.winner = self.check_winner()
        if self.winner:
            self.over = True
            return f"Player {self.winner} wins!"
        if " " not in self.board:
            self.over = True
            return "It's a draw!"
        self.current_player = "O" if self.current_player == "X" else "X"
        return "Move successful."

    def play_moves(self, moves: list[int]) -> list[str]:
        """Non-interactive replacement for a keyboard loop: play a scripted list of moves."""
        return [self.make_move(m) for m in moves]


games = {
    "X wins on the top row": ([1, 4, 2, 5, 3], "Player X wins!"),
    "draw": ([1, 2, 3, 5, 4, 6, 8, 7, 9], "It's a draw!"),
    "occupied square": ([5, 5], "Invalid move. The position is already occupied."),
}
for label, (moves, expected) in games.items():
    game = TicTacToe()
    last = game.play_moves(moves)[-1]
    print(f"{'✅' if last == expected else '❌'} {label}: moves {moves} -> {last}")
print("\nFinal board of the draw game:")
game = TicTacToe()
game.play_moves(games["draw"][0])
print(game.board_text())

---
## 📝 Summary

### 1. Critique and revise
- A **developer** (generator) and a **tester** (reflector) share one conversation; each revision sees the previous
  draft and the critique.

### 2. Running model-written code
- `run_python()` (`python_runner.py`) runs each snippet as a script in a new `python -I` process, in an empty
  temporary folder, with no environment variables, no keyboard and a timeout. It is a convenience guard, not a
  sandbox: the code still runs with your permissions.

### 3. Termination in code
- A `reflection_count` reducer, a pure `should_continue` router and a `recursion_limit` bound the loop. Stub nodes
  proved `developer = r + 1`, `tester = r` with zero model calls.

### 4. Applying it
- The pair built FLAMES and Tic-Tac-Toe in one reflection round each. The tester's scores are estimates; the
  hand-written reference implementations were checked with code-computed ✅ lines.

### Next Steps
- `12_supervisor_financial_research.ipynb`: a supervisor that routes work between a financial researcher and a coder